# Traffic Demand Prediction - Final Clean Pipeline

This notebook contains:
1. Data loading
2. Preprocessing
3. Feature engineering
4. Target encoding
5. Validation training (for model selection)
6. Full-data training
7. Submission generation


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor


In [2]:

# Load data

train = pd.read_csv('/kaggle/input/datasets/mohit78241/gridathon/dataset/train.csv')
test = pd.read_csv('/kaggle/input/datasets/mohit78241/gridathon/dataset/test.csv')


## Preprocessing & Feature Engineering

In [3]:

# Missing values

train['Temperature'] = train['Temperature'].fillna(train['Temperature'].mean())
test['Temperature'] = test['Temperature'].fillna(train['Temperature'].mean())

for col in ['RoadType', 'Weather']:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)

# Time features

train['hour'] = pd.to_datetime(train['timestamp'], format='%H:%M').dt.hour
test['hour'] = pd.to_datetime(test['timestamp'], format='%H:%M').dt.hour

train['hour_sin'] = np.sin(2 * np.pi * train['hour'] / 24)
train['hour_cos'] = np.cos(2 * np.pi * train['hour'] / 24)

test['hour_sin'] = np.sin(2 * np.pi * test['hour'] / 24)
test['hour_cos'] = np.cos(2 * np.pi * test['hour'] / 24)

# Region features

train['geo4'] = train['geohash'].str[:4]
test['geo4'] = test['geohash'].str[:4]

train['road_hour'] = (
    train['RoadType'].astype(str)
    + '_'
    + train['hour'].astype(str)
)

test['road_hour'] = (
    test['RoadType'].astype(str)
    + '_'
    + test['hour'].astype(str)
)

train['geo4_hour'] = (
    train['geo4']
    + '_'
    + train['hour'].astype(str)
)

test['geo4_hour'] = (
    test['geo4']
    + '_'
    + test['hour'].astype(str)
)


## Validation Pipeline

In [4]:

base_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'NumberofLanes',
    'Temperature',
    'hour',
    'hour_sin',
    'hour_cos'
]

X_raw = train[base_features].copy()
y = train['demand']

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42
)


In [5]:

cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

X_train = pd.get_dummies(
    X_train_raw,
    columns=cat_cols,
    drop_first=True
)

X_val = pd.get_dummies(
    X_val_raw,
    columns=cat_cols,
    drop_first=True
)

X_train, X_val = X_train.align(
    X_val,
    join='left',
    axis=1,
    fill_value=0
)


In [6]:

# Target encodings

geo_mean = pd.DataFrame({
    'geohash': X_train_raw['geohash'],
    'demand': y_train
}).groupby('geohash')['demand'].mean()

road_hour_mean = pd.DataFrame({
    'road_hour': (
        X_train_raw['RoadType'].astype(str)
        + '_'
        + X_train_raw['hour'].astype(str)
    ),
    'demand': y_train
}).groupby('road_hour')['demand'].mean()

geo4_hour_mean = pd.DataFrame({
    'geo4_hour': (
        X_train_raw['geohash'].str[:4]
        + '_'
        + X_train_raw['hour'].astype(str)
    ),
    'demand': y_train
}).groupby('geo4_hour')['demand'].mean()

global_mean = y_train.mean()


In [7]:

X_train['geo_encoded'] = X_train_raw['geohash'].map(geo_mean)
X_val['geo_encoded'] = X_val_raw['geohash'].map(geo_mean)

X_train['road_hour_encoded'] = (
    X_train_raw['RoadType'].astype(str)
    + '_'
    + X_train_raw['hour'].astype(str)
).map(road_hour_mean)

X_val['road_hour_encoded'] = (
    X_val_raw['RoadType'].astype(str)
    + '_'
    + X_val_raw['hour'].astype(str)
).map(road_hour_mean)

X_train['geo4_hour_encoded'] = (
    X_train_raw['geohash'].str[:4]
    + '_'
    + X_train_raw['hour'].astype(str)
).map(geo4_hour_mean)

X_val['geo4_hour_encoded'] = (
    X_val_raw['geohash'].str[:4]
    + '_'
    + X_val_raw['hour'].astype(str)
).map(geo4_hour_mean)

for col in ['geo_encoded', 'road_hour_encoded', 'geo4_hour_encoded']:
    X_train[col] = X_train[col].fillna(global_mean)
    X_val[col] = X_val[col].fillna(global_mean)

X_train = X_train.drop(columns=['geohash'])
X_val = X_val.drop(columns=['geohash'])


In [8]:

features_final = [
    'geo_encoded',
    'road_hour_encoded',
    'geo4_hour_encoded',
    'hour',
    'hour_sin',
    'hour_cos',
    'RoadType_Residential',
    'RoadType_Street'
]

X_train_cb = X_train[features_final]
X_val_cb = X_val[features_final]


In [9]:
cb = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=10,
    bagging_temperature=1,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=100
)

cb.fit(
    X_train_cb,
    y_train,
    eval_set=(X_val_cb, y_val),
    use_best_model=True
)


0:	learn: 0.1300049	test: 0.1300084	best: 0.1300084 (0)	total: 62.2ms	remaining: 5m 11s
100:	learn: 0.0392278	test: 0.0409462	best: 0.0409462 (100)	total: 588ms	remaining: 28.5s
200:	learn: 0.0381932	test: 0.0402531	best: 0.0402531 (200)	total: 1.11s	remaining: 26.5s
300:	learn: 0.0375887	test: 0.0398662	best: 0.0398662 (300)	total: 1.63s	remaining: 25.5s
400:	learn: 0.0371779	test: 0.0396322	best: 0.0396322 (400)	total: 2.15s	remaining: 24.7s
500:	learn: 0.0367652	test: 0.0394010	best: 0.0394010 (500)	total: 2.68s	remaining: 24.1s
600:	learn: 0.0364351	test: 0.0392172	best: 0.0392172 (600)	total: 3.2s	remaining: 23.4s
700:	learn: 0.0361627	test: 0.0390301	best: 0.0390301 (700)	total: 3.73s	remaining: 22.9s
800:	learn: 0.0359231	test: 0.0388871	best: 0.0388871 (800)	total: 4.25s	remaining: 22.3s
900:	learn: 0.0357111	test: 0.0387857	best: 0.0387857 (900)	total: 4.78s	remaining: 21.7s
1000:	learn: 0.0355376	test: 0.0386774	best: 0.0386774 (1000)	total: 5.3s	remaining: 21.2s
1100:	learn:

CatBoostRegressor(bagging_temperature=1, depth=6, early_stopping_rounds=200, eval_metric='RMSE', iterations=5000, l2_leaf_reg=10, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=100)

In [10]:
cb.get_best_iteration()

4942

In [11]:

y_pred = cb.predict(X_val_cb)

print("MAE :", mean_absolute_error(y_val, y_pred))
print("RMSE:", mean_squared_error(y_val, y_pred) ** 0.5)
print("R2  :", r2_score(y_val, y_pred))

print("Train R2:", cb.score(X_train_cb, y_train))
print("Validation R2:", cb.score(X_val_cb, y_val))


MAE : 0.023342442765449813
RMSE: 0.03771960931513574
R2  : 0.929685912708399
Train R2: 0.9456061397123444
Validation R2: 0.929685912708399


In [12]:
print(cb.get_best_iteration())

4942


## Full Data Training For Submission

In [13]:

geo_mean_full = train.groupby('geohash')['demand'].mean()

road_hour_mean_full = pd.DataFrame({
    'road_hour': train['road_hour'],
    'demand': train['demand']
}).groupby('road_hour')['demand'].mean()

geo4_hour_mean_full = pd.DataFrame({
    'geo4_hour': train['geo4_hour'],
    'demand': train['demand']
}).groupby('geo4_hour')['demand'].mean()

global_mean_full = train['demand'].mean()


In [14]:

X_full = pd.get_dummies(
    train[base_features],
    columns=['RoadType', 'LargeVehicles', 'Landmarks', 'Weather'],
    drop_first=True
)

X_full['geo_encoded'] = train['geohash'].map(geo_mean_full)
X_full['road_hour_encoded'] = train['road_hour'].map(road_hour_mean_full)
X_full['geo4_hour_encoded'] = train['geo4_hour'].map(geo4_hour_mean_full)

for col in ['geo_encoded', 'road_hour_encoded', 'geo4_hour_encoded']:
    X_full[col] = X_full[col].fillna(global_mean_full)

X_full = X_full.drop(columns=['geohash'])

X_full = X_full[features_final]

y_full = train['demand']


In [15]:

X_test = pd.get_dummies(
    test[base_features],
    columns=['RoadType', 'LargeVehicles', 'Landmarks', 'Weather'],
    drop_first=True
)

X_test['geo_encoded'] = test['geohash'].map(geo_mean_full)
X_test['road_hour_encoded'] = test['road_hour'].map(road_hour_mean_full)
X_test['geo4_hour_encoded'] = test['geo4_hour'].map(geo4_hour_mean_full)

for col in ['geo_encoded', 'road_hour_encoded', 'geo4_hour_encoded']:
    X_test[col] = X_test[col].fillna(global_mean_full)

X_test = X_test.drop(columns=['geohash'])

X_test = X_test.reindex(columns=X_full.columns, fill_value=0)
X_test = X_test[features_final]


In [16]:
cb_final = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=10,
    bagging_temperature=1,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=100
)

cb_final.fit(
    X_full,
    y_full
)


0:	learn: 0.1300109	total: 7.61ms	remaining: 38s
100:	learn: 0.0393149	total: 615ms	remaining: 29.8s
200:	learn: 0.0378960	total: 1.22s	remaining: 29.1s
300:	learn: 0.0372091	total: 1.82s	remaining: 28.4s
400:	learn: 0.0367422	total: 2.47s	remaining: 28.3s
500:	learn: 0.0363486	total: 3.08s	remaining: 27.7s
600:	learn: 0.0360118	total: 3.7s	remaining: 27.1s
700:	learn: 0.0357524	total: 4.33s	remaining: 26.5s
800:	learn: 0.0355233	total: 4.95s	remaining: 25.9s
900:	learn: 0.0353269	total: 5.63s	remaining: 25.6s
1000:	learn: 0.0351390	total: 6.25s	remaining: 25s
1100:	learn: 0.0349652	total: 6.86s	remaining: 24.3s
1200:	learn: 0.0348191	total: 7.48s	remaining: 23.7s
1300:	learn: 0.0346889	total: 8.1s	remaining: 23s
1400:	learn: 0.0345803	total: 8.71s	remaining: 22.4s
1500:	learn: 0.0344621	total: 9.32s	remaining: 21.7s
1600:	learn: 0.0343504	total: 9.93s	remaining: 21.1s
1700:	learn: 0.0342520	total: 10.6s	remaining: 20.5s
1800:	learn: 0.0341594	total: 11.2s	remaining: 19.8s
1900:	learn:

CatBoostRegressor(bagging_temperature=1, depth=6, early_stopping_rounds=200, eval_metric='RMSE', iterations=5000, l2_leaf_reg=10, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=100)

In [17]:

test_preds = cb_final.predict(X_test)

submission = pd.DataFrame({
    'Index': range(len(test_preds)),
    'demand': test_preds
})

submission.to_csv('submission.csv', index=False)

print(submission.shape)
submission.head()


(41778, 2)


,Index,demand
0,0,0.040804
1,1,0.026923
2,2,0.033872
3,3,0.040804
4,4,0.058132
